# ทดสอบครบเส้นทาง · API จริง + Gemini จริง + ส่งต่อ Agent 2 ทาง HTTP

รันจากบนลงล่าง (Run All) Notebook นี้ทดสอบว่า Agent 1 ทำได้จริงครบ 5 อย่าง:

| ความสามารถ | ทดสอบอย่างไร |
|---|---|
| **รับเรื่อง** | ผู้ใช้แจ้งน้ำท่วม → ระบบสร้างเคส draft และถามต่อ |
| **จดจำข้อมูล** | ตอบหลายข้อความ → สรุปรวมครบ · **ปิดแล้วเปิด API ใหม่** → ยังจำเคสเดิมจากฐานข้อมูล |
| **จัดการเคส** | ยืนยัน → submitted · ข้อมูลเปลี่ยน → incident_updated · priority ตามกฎ |
| **ตอบโต้** | คำตอบภาษาไทยจาก Gemini + template · ถามสถานะแล้วตอบตามจริง |
| **ส่งต่อ** | Agent 2 จำลองรับ event ทาง HTTP จริงและตอบ ACK · เคสด่วนส่งทันที · สรุปพื้นที่ |

- เรียก `POST /chat` ของ `app.main` ตัวเดียวกับ server ผ่าน `TestClient` (lifespan, background worker, outbox, retry ทำงานจริง)
- ฐานข้อมูล SQLite ในโฟลเดอร์ชั่วคราว ลบได้ ไม่แตะ `agent1.sqlite3`
- เรียก Gemini จริงประมาณ 10–20 ครั้งต่อรอบ มีค่าใช้จ่ายตาม key ใน `dspy-agent/.env` หรือ `lab/1/.env`
- ใช้ข้อมูลสมมติเท่านั้น รายชื่อพื้นที่ยัง `pending_review` จึงขึ้น `unverified` และสรุปพื้นที่อยู่กลุ่ม `unassigned` ตามจริง
- คำตอบของ LLM เปลี่ยนได้ทุกรอบ ผลตรวจท้าย Notebook จึงบันทึกผ่าน/ไม่ผ่าน ไม่หยุดกลางทาง

ถ้า kernel ยังไม่มี dependency: `%pip install -r /Users/bubboo/DSI490/dspy-agent/requirements-notebook.txt` แล้ว Restart Kernel

In [1]:
import os
import sys
import json
import time
import tempfile
import threading
from datetime import datetime, timezone, timedelta
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path

import dspy
from dotenv import load_dotenv, dotenv_values

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dspy-agent").is_dir())
APP_ROOT = ROOT / "dspy-agent"
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))
load_dotenv(APP_ROOT / ".env", override=False)
load_dotenv(ROOT / "lab" / "1" / ".env", override=False)

api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY") or os.getenv("API_KEY")
model = (dotenv_values(ROOT / "lab" / "1" / ".env").get("GEMINI_MODEL")
         or dotenv_values(APP_ROOT / ".env").get("GEMINI_MODEL")
         or os.getenv("GEMINI_MODEL") or "gemini/gemini-3.1-flash-lite").strip()
if "/" not in model:
    model = "gemini/" + model
if not api_key:
    raise RuntimeError("ตั้ง GEMINI_API_KEY ใน dspy-agent/.env หรือ lab/1/.env ก่อน")
# API จะใช้ LM ที่ตั้งไว้แล้วนี้ (configure_dspy ข้ามการตั้งซ้ำ)
dspy.configure(lm=dspy.LM(model=model, api_key=api_key, cache=False, timeout=60, num_retries=2))
print("model =", model)

model = gemini/gemini-3.1-flash-lite


## Agent 2 จำลอง

HTTP server เล็ก ๆ รับ `POST /ingest` ตรวจ token บันทึกทุก event แล้วตอบ Technical ACK `{event_id, received, received_at}` ตามสัญญาเดียวกับ Agent 2 จริง

In [2]:
INBOX = []          # event ที่ Agent 2 ได้รับ ตามลำดับ
TOKEN = "demo-token"


class Agent2Stub(BaseHTTPRequestHandler):
    def do_POST(self):
        if self.path != "/ingest" or self.headers.get("Authorization") != "Bearer " + TOKEN:
            self.send_response(403); self.end_headers(); return
        event = json.loads(self.rfile.read(int(self.headers["Content-Length"])))
        INBOX.append(event)
        ack = json.dumps({"event_id": event["event_id"], "received": True,
                          "received_at": datetime.now(timezone(timedelta(hours=7))).isoformat()}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(ack)))
        self.end_headers()
        self.wfile.write(ack)

    def log_message(self, *args):
        pass


try:
    agent2_server.shutdown()  # รันเซลล์ซ้ำได้
except NameError:
    pass
agent2_server = ThreadingHTTPServer(("127.0.0.1", 0), Agent2Stub)
threading.Thread(target=agent2_server.serve_forever, daemon=True).start()

DB_DIR = Path(tempfile.mkdtemp(prefix="agent1_e2e_"))
os.environ["DATABASE_URL"] = f"sqlite:///{DB_DIR / 'agent1.sqlite3'}"
os.environ["AGENT2_HANDOFF_URL"] = f"http://127.0.0.1:{agent2_server.server_port}/ingest"
os.environ["AGENT2_HANDOFF_TOKEN"] = TOKEN
print("Agent 2 stub:", os.environ["AGENT2_HANDOFF_URL"])
print("database   :", os.environ["DATABASE_URL"])

Agent 2 stub: http://127.0.0.1:54322/ingest
database   : sqlite:////var/folders/wd/pzn4r40s7y72b4dz1mxtjv8h0000gn/T/agent1_e2e_aq_asky_/agent1.sqlite3


## ตัวช่วย

`start_api()` เปิด API (รัน lifespan จริง) · `stop_api()` ปิด · `say()` ส่งข้อความเข้า `/chat` · `check()` บันทึกผลตรวจ · `show_inbox()` ดูสิ่งที่ Agent 2 ได้รับ

In [3]:
from fastapi.testclient import TestClient
from app.main import app

client = None
CHECKS = []


def start_api():
    global client
    client = TestClient(app)
    client.__enter__()          # lifespan: โหลด config, repository, background worker
    print("API เปิดแล้ว:", client.get("/health").json())


def stop_api():
    global client
    client.__exit__(None, None, None)
    client = None
    print("API ปิดแล้ว (หน่วยความจำของ process ถูกล้าง เหลือแค่ฐานข้อมูล)")


def say(user_id, text=None, postback=None):
    body = {"user_id": user_id, **({"postback": postback} if postback else {"input_text": text})}
    r = client.post("/chat", json=body)
    r.raise_for_status()
    out = r.json()
    print(f"\n👤 {text or 'กดปุ่ม: ' + postback}")
    print("🤖 " + out["output_text"].replace("\n", "\n   "))
    print(f"   ⚙️ incident={out['incident_id']} status={out['incident_status']} next={out['next_actions']}")
    return out


def check(name, ok, detail=""):
    CHECKS.append((name, bool(ok), detail))
    print(("✅" if ok else "❌"), name, ("— " + detail) if detail else "")


def events(kind=None, incident_id=None):
    return [e for e in INBOX if (kind is None or e["event_type"] == kind)
            and (incident_id is None or e.get("incident_id") == incident_id)]


def show_inbox():
    print(f"Agent 2 ได้รับ {len(INBOX)} event")
    for e in INBOX:
        if e["event_type"] == "area_summary_updated":
            s = e["summary"]
            print(f"  · area_summary_updated {s['area_level']}:{s['area_code']} เคส={list(s['case_versions'])} "
                  f"ด่วน={s['urgent_incident_ids']} {s['priority_counts']} needs={s['needs']}")
            continue
        r = e.get("report") or {}
        print(f"  · {e['event_type']} {e.get('incident_id')} v{e.get('case_version')} {e.get('trigger', '')} "
              f"priority={(r.get('priority') or {}).get('level')} flags={r.get('urgent_flags')}")
        for c in e.get("changes", []):
            print(f"      change {c['field']} ({c['change_type']})")


def wait_for(predicate, seconds=10):
    end = time.time() + seconds
    while time.time() < end and not predicate():
        time.sleep(0.5)
    return predicate()


start_api()

hotlines_th.json ยังไม่ได้ตรวจกับแหล่งทางการ (status=draft-unverified)


API เปิดแล้ว: {'status': 'ok', 'model': 'gemini/gemini-3.1-flash-lite'}


## 1. รับเรื่อง

ผู้ใช้ A แจ้งน้ำท่วมทั่วไป (ไม่ด่วน) ระบบควรสร้างเคส `draft` และถามข้อมูลต่อ ยังไม่ส่งต่อ

In [4]:
A = "user_demo_a"
out = say(A, "น้ำท่วมเข้าบ้านแล้วค่ะ สูงประมาณเข่า อยู่ตำบลน้ำตาล อำเภออินทร์บุรี")
case_a = out["incident_id"]
check("รับเรื่อง: สร้างเคส draft", case_a and out["incident_status"] == "draft")
check("รับเรื่อง: ถามข้อมูลต่อ", "ask_follow_up" in out["next_actions"])
check("รับเรื่อง: เหตุทั่วไปยังไม่ส่ง Agent 2", not events(incident_id=case_a))


👤 น้ำท่วมเข้าบ้านแล้วค่ะ สูงประมาณเข่า อยู่ตำบลน้ำตาล อำเภออินทร์บุรี
🤖 รับทราบข้อมูลที่แจ้งค่ะ
   
   จุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ
   
   ระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ
   ⚙️ incident=inc_fe53abf12329 status=draft next=['ask_follow_up']
✅ รับเรื่อง: สร้างเคส draft 
✅ รับเรื่อง: ถามข้อมูลต่อ 
✅ รับเรื่อง: เหตุทั่วไปยังไม่ส่ง Agent 2 


## 2. จดจำข้อมูลข้ามข้อความ

ข้ามการแชร์ตำแหน่ง แล้วให้ข้อมูลหลายอย่างในข้อความเดียว ระบบควรรวมกับที่บอกไว้ก่อนหน้า แล้วสรุปให้ยืนยัน

In [5]:
say(A, postback="location_share=skip")
out = say(A, "อยู่หลังวัดน้ำตาลค่ะ อยากได้น้ำดื่มกับอาหาร บ้านมี 4 คน มียายอายุ 80 อยู่ด้วย 1 คน")
text = out["output_text"]
check("จดจำ: ใช้เคสเดิม", out["incident_id"] == case_a)
check("จดจำ: สรุปจำนวนคน 4 คน", "4 คน" in text)
check("จดจำ: จำผู้สูงอายุ", "ผู้สูงอายุ" in text)
check("จดจำ: จำระดับน้ำจากข้อความแรก", "เข่า" in text)
check("จัดการเคส: ขอให้ยืนยันก่อนส่ง", "confirm_report" in out["next_actions"])


👤 กดปุ่ม: location_share=skip
🤖 รับทราบว่าไม่สะดวกให้ข้อมูลส่วนนี้ค่ะ
   
   ตอนนี้ต้องการความช่วยเหลือเรื่องใด หรือเพียงแจ้งให้ทราบ กรุณาบอกได้เลยค่ะ
   ⚙️ incident=inc_fe53abf12329 status=draft next=['ask_follow_up']

👤 อยู่หลังวัดน้ำตาลค่ะ อยากได้น้ำดื่มกับอาหาร บ้านมี 4 คน มียายอายุ 80 อยู่ด้วย 1 คน
🤖 รับทราบข้อมูลที่แจ้งค่ะ
   
   สรุปข้อมูลที่แจ้งค่ะ
   - พื้นที่: ตำบลน้ำตาล อำเภออินทร์บุรี
   - จุดสังเกต: หลังวัดน้ำตาล
   - ตำแหน่ง: ไม่สะดวกให้ข้อมูล
   - สถานการณ์: น้ำท่วมเข้าบ้านแล้วค่ะ สูงประมาณเข่า อยู่ตำบลน้ำตาล อำเภออินทร์บุรี
   - ความต้องการ: อาหารและน้ำดื่ม, สิ่งของจำเป็น
   - ผู้ได้รับผลกระทบ: 4 คน
   - ผู้ที่ต้องดูแลเป็นพิเศษ: ผู้สูงอายุ 1 คน
   - ระดับน้ำ: สูงประมาณเข่า
   หากข้อมูลถูกต้อง กรุณายืนยันให้ส่งรายงาน หรือแจ้งส่วนที่ต้องการแก้ไขได้ค่ะ
   ⚙️ incident=inc_fe53abf12329 status=draft next=['confirm_report']
✅ จดจำ: ใช้เคสเดิม 
✅ จดจำ: สรุปจำนวนคน 4 คน 
✅ จดจำ: จำผู้สูงอายุ 
✅ จดจำ: จำระดับน้ำจากข้อความแรก 
✅ จัดการเคส: ขอให้ยืนยันก่อนส่ง 


## 3. ยืนยัน → ส่งต่อ Agent 2 และถามสถานะ

หลังยืนยัน เคสเป็น `submitted` และ Agent 2 ได้รับ `incident_submitted` ทาง HTTP ระบบต้องตอบสถานะตามจริง ไม่อ้างว่าเจ้าหน้าที่รับเรื่องแล้ว

In [6]:
out = say(A, "ถูกต้องค่ะ ยืนยันส่งเลย")
check("จัดการเคส: submitted หลังยืนยัน", out["incident_status"] == "submitted")
check("ส่งต่อ: Agent 2 ได้รับ incident_submitted",
      wait_for(lambda: events("incident_submitted", case_a)))

out = say(A, "ส่งเรื่องไปถึงเจ้าหน้าที่หรือยังคะ")
check("ตอบโต้: ไม่อ้างว่าเจ้าหน้าที่รับเรื่องแล้ว", "เจ้าหน้าที่รับเรื่องแล้ว" not in out["output_text"])
show_inbox()


👤 ถูกต้องค่ะ ยืนยันส่งเลย
🤖 บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ
   ⚙️ incident=inc_fe53abf12329 status=submitted next=['submit_report']
✅ จัดการเคส: submitted หลังยืนยัน 
✅ ส่งต่อ: Agent 2 ได้รับ incident_submitted 

👤 ส่งเรื่องไปถึงเจ้าหน้าที่หรือยังคะ
🤖 บันทึกและส่งข้อมูลให้ระบบประสานงานแล้วค่ะ
   ⚙️ incident=inc_fe53abf12329 status=submitted next=['none']
✅ ตอบโต้: ไม่อ้างว่าเจ้าหน้าที่รับเรื่องแล้ว 
Agent 2 ได้รับ 1 event
  · incident_submitted inc_fe53abf12329 v1 user_confirmed priority=yellow flags=[]


## 4. ปิด API แล้วเปิดใหม่ → ยังจำเคสเดิม

จำลอง server รีสตาร์ต หน่วยความจำของ process หายหมด ระบบต้องโหลดเคสจากฐานข้อมูล แล้วส่ง `incident_updated` เมื่อผู้ใช้แจ้งข้อมูลใหม่

In [7]:
stop_api()
start_api()

out = say(A, "ตอนนี้น้ำขึ้นถึงเอวแล้วค่ะ ยายเดินไม่ไหว")
check("จดจำหลังรีสตาร์ต: ใช้เคสเดิม", out["incident_id"] == case_a)
check("ส่งต่อ: Agent 2 ได้รับ incident_updated",
      wait_for(lambda: events("incident_updated", case_a)))
upd = events("incident_updated", case_a)
if upd:
    fields = [c["field"] for c in upd[-1]["changes"]]
    check("จัดการเคส: changes มีระดับน้ำ", "water_level" in fields, str(fields))
    check("จัดการเคส: เวอร์ชันเคสเพิ่มเป็น 2+", upd[-1]["case_version"] >= 2)

hotlines_th.json ยังไม่ได้ตรวจกับแหล่งทางการ (status=draft-unverified)


API ปิดแล้ว (หน่วยความจำของ process ถูกล้าง เหลือแค่ฐานข้อมูล)
API เปิดแล้ว: {'status': 'ok', 'model': 'gemini/gemini-3.1-flash-lite'}

👤 ตอนนี้น้ำขึ้นถึงเอวแล้วค่ะ ยายเดินไม่ไหว
🤖 จากที่แจ้งว่ามีคนติดค้าง กรุณาติดต่อช่องทางฉุกเฉินเพื่อขอความช่วยเหลือทันทีค่ะ
   • สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
   • การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669
   
   บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ
   
   หากสะดวก สามารถฝากเบอร์ไว้ให้ผู้ดูแลรายงานใช้สอบถามรายละเอียดเหตุนี้เพื่อประสานความช่วยเหลือได้ค่ะ สามารถข้ามได้ และระบบจะขอยืนยันอนุญาตใช้เบอร์แยกอีกครั้งค่ะ
   ⚙️ incident=inc_fe53abf12329 status=submitted next=['send_update', 'ask_follow_up']
✅ จดจำหลังรีสตาร์ต: ใช้เคสเดิม 
✅ ส่งต่อ: Agent 2 ได้รับ incident_updated 
✅ จัดการเคส: changes มีระดับน้ำ — ['situation', 'water_level', 'urgent_flags', 'priority']
✅ จัดการเคส: เวอร์ชันเคสเพิ่มเป็น 2+ 


## 5. เคสด่วน → ส่งทันที ไม่ต้องยืนยัน

ผู้ใช้ B แจ้งคนติดบนหลังคา ระบบควรแสดงสายด่วน ส่ง `incident_submitted` ทันทีด้วย trigger `urgent_flags` และ priority red แล้วค่อยถามต่อ

In [8]:
B = "user_demo_b"
out = say(B, "ช่วยด้วยค่ะ พ่อติดอยู่บนหลังคา น้ำขึ้นเร็วมาก บ้านอยู่ตำบลทับยา")
case_b = out["incident_id"]
check("เคสด่วน: submitted ทันที", out["incident_status"] == "submitted")
check("ตอบโต้: แสดงสายด่วน", "1784" in out["output_text"] or "1669" in out["output_text"])
check("ส่งต่อ: ส่งทันทีด้วย urgent_flags",
      wait_for(lambda: any(e.get("trigger") == "urgent_flags" for e in events("incident_submitted", case_b))))
sub_b = events("incident_submitted", case_b)
if sub_b:
    check("จัดการเคส: priority red", sub_b[0]["report"]["priority"]["level"] == "red")

say(B, "มีพ่อคนเดียวค่ะ อายุ 70 เป็นเบาหวาน")


👤 ช่วยด้วยค่ะ พ่อติดอยู่บนหลังคา น้ำขึ้นเร็วมาก บ้านอยู่ตำบลทับยา
🤖 จากข้อมูลที่แจ้ง กรุณาติดต่อช่องทางฉุกเฉินเพื่อขอความช่วยเหลือทันทีค่ะ
   • สายด่วน ปภ. (ยังไม่ได้ตรวจ) 1784
   • การแพทย์ฉุกเฉิน (ยังไม่ได้ตรวจ) 1669
   
   บันทึกและส่งข้อมูลให้ระบบประสานงานแล้วค่ะ
   
   จุดเกิดเหตุอยู่ที่ไหนคะ? แชร์ตำแหน่ง หรือพิมพ์หมู่บ้าน ตำบล และจุดสังเกตได้ค่ะ หากแจ้งแทนผู้อื่น กรุณาระบุตำแหน่งของจุดเกิดเหตุ ตำแหน่งจะใช้ระบุจุดเกิดเหตุและแนบรายงานให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือ หากไม่สะดวก สามารถข้ามได้ค่ะ
   
   ระบบจะใช้ข้อมูลที่คุณแจ้งเพื่อจัดทำรายงานและส่งต่อให้ผู้ดูแลรายงานเพื่อประสานความช่วยเหลือค่ะ
   ⚙️ incident=inc_5553665d9573 status=submitted next=['submit_report', 'ask_follow_up']
✅ เคสด่วน: submitted ทันที 
✅ ตอบโต้: แสดงสายด่วน 
✅ ส่งต่อ: ส่งทันทีด้วย urgent_flags 
✅ จัดการเคส: priority red 

👤 มีพ่อคนเดียวค่ะ อายุ 70 เป็นเบาหวาน
🤖 บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ
   
   ขอจุดสังเกตใกล้ที่เกิดเหตุ เช่น วัด โรงเรียน หรือชื่อถนนค่ะ
   ⚙️ incident=inc_5553665d9573 status=

{'output_text': 'บันทึกข้อมูลไว้แล้ว กำลังส่งต่อค่ะ\n\nขอจุดสังเกตใกล้ที่เกิดเหตุ เช่น วัด โรงเรียน หรือชื่อถนนค่ะ',
 'intents': ['update_incident'],
 'incident_id': 'inc_5553665d9573',
 'incident_status': 'submitted',
 'matching_status': 'new',
 'urgent_flags': ['help_requested', 'people_trapped'],
 'missing_fields': [],
 'next_actions': ['send_update', 'ask_follow_up'],
 'pending_question': {'incident_id': 'inc_5553665d9573',
  'field_names': ['location_landmark'],
  'asked_at': '2026-11-01T23:54:15.562334+07:00',
  'asked_message_id': 'msg_01ceb5debee4_reply'},
 'delivery_status': 'pending',
 'sources': [],
 'used_hotline_template': False,
 'quick_replies': [],
 'model': 'gemini/gemini-3.1-flash-lite'}

## 6. สรุปภาพรวมพื้นที่

background worker สร้างสรุปพื้นที่ทุก 30 วินาที แล้วส่ง `area_summary_updated` ให้ Agent 2 ต้องมีทั้งสองเคส และมีเคสด่วนใน `urgent_incident_ids` (รอไม่เกิน 45 วินาที)

In [9]:
def summary_has_both():
    return any(set(e["summary"]["case_versions"]) >= {case_a, case_b} for e in events("area_summary_updated"))

check("ส่งต่อ: Agent 2 ได้รับสรุปพื้นที่ที่มีทั้งสองเคส", wait_for(summary_has_both, 45))
latest = [e for e in events("area_summary_updated") if set(e["summary"]["case_versions"]) >= {case_a, case_b}]
if latest:
    s = latest[-1]["summary"]
    check("สรุปพื้นที่: เคสด่วนอยู่ใน urgent_incident_ids",
          case_b in s["urgent_incident_ids"], str(s["urgent_incident_ids"]))
    # เคส A อาจอยู่ในรายการด้วย ถ้า Gemini ตั้ง flag จากข้อความ "ยายเดินไม่ไหว" (ส่งเกินดีกว่าพลาด ข้อ 6.3)
show_inbox()

✅ ส่งต่อ: Agent 2 ได้รับสรุปพื้นที่ที่มีทั้งสองเคส 
✅ สรุปพื้นที่: เคสด่วนอยู่ใน urgent_incident_ids — ['inc_5553665d9573', 'inc_fe53abf12329']
Agent 2 ได้รับ 5 event
  · incident_submitted inc_fe53abf12329 v1 user_confirmed priority=yellow flags=[]
  · incident_updated inc_fe53abf12329 v2  priority=red flags=['people_trapped']
      change situation (update)
      change water_level (update)
      change urgent_flags (update)
      change priority (update)
  · incident_submitted inc_5553665d9573 v1 urgent_flags priority=red flags=['help_requested', 'people_trapped']
  · incident_updated inc_5553665d9573 v2  priority=red flags=['help_requested', 'people_trapped']
      change vulnerable_people (initial)
  · area_summary_updated unassigned:None เคส=['inc_fe53abf12329', 'inc_5553665d9573'] ด่วน=['inc_5553665d9573', 'inc_fe53abf12329'] {'red': 2} needs=['food_water', 'supplies']


## 7. ผลตรวจ และสิ่งที่ระบบเก็บ

ACK ทุก event ต้องเป็น `acked` ใน outbox · ตรวจได้ว่าระบบเก็บอะไรไว้จริงในฐานข้อมูล

In [10]:
from app.schemas import DeliveryStatus

repo = client.app.state.repository
outbox = repo.outbox()
check("ส่งต่อ: ทุก event ใน outbox ได้ ACK",
      outbox and all(r.delivery_status == DeliveryStatus.ACKED for r in outbox),
      f"{sum(r.delivery_status == DeliveryStatus.ACKED for r in outbox)}/{len(outbox)} acked")

print()
passed = sum(ok for _, ok, _ in CHECKS)
print(f"ผ่าน {passed}/{len(CHECKS)} ข้อ")
for name, ok, detail in CHECKS:
    if not ok:
        print("❌", name, detail)

✅ ส่งต่อ: ทุก event ใน outbox ได้ ACK — 5/5 acked

ผ่าน 22/22 ข้อ


In [11]:
# ดู state ของเคส A ที่เก็บไว้ (ไม่มีเบอร์โทรหรือ LINE ID อยู่ในรายงาน)
print(json.dumps(events("incident_updated", case_a)[-1]["report"], ensure_ascii=False, indent=2)[:3000])

{
  "incident_id": "inc_fe53abf12329",
  "incident_version": 2,
  "reporter_ref": "rep_af2aadf50f17",
  "source": "line_oa",
  "status": "submitted",
  "matching_status": "new",
  "possible_related_incident_ids": [],
  "related_incident_id": null,
  "service_scope": "unverified",
  "location": {
    "area": {
      "raw": "ตำบลน้ำตาล อำเภออินทร์บุรี",
      "stated_level": "unspecified",
      "match_status": "unverified",
      "tambon_code": null,
      "tambon": null,
      "amphoe_code": null,
      "amphoe": null,
      "province_code": null,
      "province": null,
      "candidates": []
    },
    "landmark": "หลังวัดน้ำตาล",
    "coordinates": null,
    "coordinates_basis": null
  },
  "location_missing": false,
  "case_version": 2,
  "priority": {
    "level": "red",
    "reasons": [
      "people_trapped"
    ],
    "evaluated_at": "2026-11-01T23:54:08.512808+07:00",
    "rules_version": "draft-1",
    "source": "agent1_rules",
    "adjusted_by": null
  },
  "situation": {
  

In [12]:
# ปิด API และ Agent 2 จำลอง (ฐานข้อมูลชั่วคราวอยู่ใน DB_DIR ลบทิ้งได้)
stop_api()
agent2_server.shutdown()
print("ปิดแล้ว · ฐานข้อมูล:", DB_DIR)

API ปิดแล้ว (หน่วยความจำของ process ถูกล้าง เหลือแค่ฐานข้อมูล)
ปิดแล้ว · ฐานข้อมูล: /var/folders/wd/pzn4r40s7y72b4dz1mxtjv8h0000gn/T/agent1_e2e_aq_asky_
